# TimeGPT univariate baseline (h in {1, 6, 12})

Upstream source: `Univariate_horizon.ipynb` in the thesis workbench.

**Input**: the first non-empty folder among `DTW_DFS_DIR / 'dtw_{N}_dfs'` (the univariate baseline only needs `(week, cpc_week)`, so any DTW-N panel works).
**Output**: `timegpt_results_dir('semantic_univariate_models')` - `timegpt_univariate_metrics_h1_h6_h12.csv`, `timegpt_univariate_skipped.csv`.

> **Note**: the original filename is `Univariate_horizon.ipynb`, which suggests a Prophet baseline. In practice the notebook runs **TimeGPT** (Prophet univariate is covered inside `prophet/01_prophet_dtw_multiN.ipynb` as the `univariate` column). The leftover CmdStan install cell from the Prophet-era draft has been removed.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path('..', '_shared').resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import (  # noqa: E402
    ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_5, timegpt_results_dir,
)
from api_keys import get_nixtla_key  # noqa: E402

ensure_env()
os.environ['NIXTLA_API_KEY'] = get_nixtla_key()


In [ ]:
# ==================== TimeGPT UNIVARIATE (h=1,6,12) — per keyword ====================
!pip -q install nixtla polars pyarrow tqdm

import os, re, warnings
from pathlib import Path
from datetime import date
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from nixtla import NixtlaClient

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option('future.no_silent_downcasting', True)

# ---------------- Paths & config ----------------
BASE = DATA_ROOT

# choose ANY folder that contains the keyword parquet files with (week, cpc_week).
# We'll auto-pick the first existing dtw_* folder with parquet files.
CANDIDATE_INPUT_DIRS = [
    BASE / "dtw_neighbour_dfs" / "dtw_7_dfs",
    BASE / "dtw_neighbour_dfs" / "dtw_10_dfs",
    BASE / "dtw_neighbour_dfs" / "dtw_20_dfs",
    BASE / "dtw_neighbour_dfs" / "dtw_2_dfs",
    BASE / "dtw_neighbour_dfs" / "dtw_3_dfs",
    BASE / "dtw_neighbour_dfs" / "dtw_5_dfs",
]

INPUT_DIR = None
for d in CANDIDATE_INPUT_DIRS:
    if d.exists() and any(d.glob("*.parquet")):
        INPUT_DIR = d
        break

assert INPUT_DIR is not None, f"No parquet files found in candidate dirs under {BASE / 'dtw_neighbour_dfs'}"

OUT_DIR = timegpt_results_dir("semantic_univariate_models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FREQ = "W-MON"
HORIZONS = [1, 6, 12]
MIN_EXTRA_WEEKS = 5  # need at least h + MIN_EXTRA_WEEKS after gridding to be safe

OUT_METRICS_CSV = OUT_DIR / "timegpt_univariate_metrics_h1_h6_h12.csv"
OUT_SKIPPED_CSV = OUT_DIR / "timegpt_univariate_skipped.csv"

# ---------------- Auth (robust secret loading) ----------------
# NIXTLA_API_KEY is set by the bootstrap cell via get_nixtla_key().

assert os.environ.get("NIXTLA_API_KEY"), (
    "NIXTLA_API_KEY not found. Copy .env.example to .env and fill it in."
)
client = NixtlaClient(api_key=os.environ["NIXTLA_API_KEY"])

# ---------------- Helpers ----------------
def iso_to_date_robust(val):
    """Parse week labels like 'WW-YYYY' or 'YYYY-WW' into ISO Monday dates."""
    if hasattr(val, "isocalendar"):
        iso = val.isocalendar()
        return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    m1 = re.fullmatch(r"(?P<wk>\d{1,2})-(?P<yr>\d{4})", s)  # WW-YYYY
    if m1:
        wk, yr = int(m1["wk"]), int(m1["yr"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    m2 = re.fullmatch(r"(?P<yr>\d{4})-(?P<wk>\d{1,2})", s)  # YYYY-WW
    if m2:
        yr, wk = int(m2["yr"]), int(m2["wk"])
        if 1 <= wk <= 53:
            return date.fromisocalendar(yr, wk, 1)
    try:
        ts = pd.to_datetime(s, errors="coerce")
        if pd.notna(ts):
            iso = ts.isocalendar()
            return date.fromisocalendar(int(iso.year), int(iso.week), 1)
    except Exception:
        pass
    return None

def build_weekly_grid(pdf: pd.DataFrame, freq=FREQ) -> pd.DataFrame:
    """Dedupe by ds via mean, then reindex to weekly grid."""
    if pdf.empty:
        return pdf
    pdf = pdf.sort_values("ds")
    pdf = pdf.groupby("ds", as_index=False).mean(numeric_only=True)
    idx = pd.date_range(pdf["ds"].min(), pdf["ds"].max(), freq=freq)
    out = pdf.set_index("ds").reindex(idx)
    out.index.name = "ds"
    return out.reset_index()

def impute_train_y_no_leak(tr: pd.DataFrame) -> pd.DataFrame:
    """Interpolate y in train only, no peeking at test."""
    tr = tr.copy().set_index("ds")
    tr["y"] = pd.to_numeric(tr["y"], errors="coerce")
    tr["y"] = tr["y"].interpolate(method="time", limit_direction="both").ffill().bfill()
    return tr.reset_index()

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    return float(100 * np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask]))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def preprocess_keyword_file_univariate(parquet_path: Path):
    """Returns gridded pdf with columns: ds, y or (None, reason_tuple)."""
    try:
        df_pl = pl.read_parquet(parquet_path)
    except Exception as e:
        return None, ("fatal_read", str(e))

    raw_cols = df_pl.columns
    if "week" not in raw_cols or "cpc_week" not in raw_cols:
        return None, ("missing_cols", f"has={raw_cols}")

    df_pl = (
        df_pl.select(["week", "cpc_week"])
             .with_columns(pl.col("week").map_elements(iso_to_date_robust, return_dtype=pl.Date).alias("ds"))
             .drop("week")
             .filter(pl.col("ds").is_not_null())
             .sort("ds")
    )

    pdf0 = df_pl.to_pandas().rename(columns={"cpc_week": "y"})
    pdf0["y"] = pd.to_numeric(pdf0["y"], errors="coerce")

    pdf = build_weekly_grid(pdf0, FREQ)

    return pdf, None

# ---------------- Run univariate per keyword ----------------
files = sorted(INPUT_DIR.glob("*.parquet"))
print(f"Using input dir: {INPUT_DIR}")
print(f"Found {len(files)} parquet files.")

metrics_rows = []
skipped_rows = []
reasons = Counter()

for p in tqdm(files, desc="TimeGPT univariate", leave=False):
    kw = p.stem
    pdf, err = preprocess_keyword_file_univariate(p)
    if pdf is None:
        reasons[err[0]] += 1
        skipped_rows.append({"keyword": kw, "reason": err[0], "info": err[1]})
        continue

    # Ensure enough length for max horizon
    max_h = max(HORIZONS)
    if len(pdf) <= max_h + MIN_EXTRA_WEEKS:
        reasons["too_short"] += 1
        skipped_rows.append({"keyword": kw, "reason": "too_short", "info": len(pdf)})
        continue

    row = {"keyword": kw, "n_obs": int(len(pdf))}

    # Run each horizon separately (new split each time)
    for h in HORIZONS:
        split = len(pdf) - h
        tr = pdf.iloc[:split].copy()
        te = pdf.iloc[split:].copy()

        # can't score if test y has NaN
        if te["y"].isna().any():
            reasons[f"nan_in_test_y_h{h}"] += 1
            row[f"smape_h{h}"] = np.nan
            row[f"rmse_h{h}"] = np.nan
            continue

        tr_imp = impute_train_y_no_leak(tr)
        if tr_imp["y"].isna().any():
            reasons[f"nan_remain_in_train_y_h{h}"] += 1
            row[f"smape_h{h}"] = np.nan
            row[f"rmse_h{h}"] = np.nan
            continue

        df_tr = tr_imp[["ds", "y"]].copy()
        df_tr["unique_id"] = kw
        df_te = te[["ds", "y"]].copy()
        df_te["unique_id"] = kw

        try:
            fc = client.forecast(
                df=df_tr[["unique_id", "ds", "y"]],
                h=h,
                freq=FREQ,
                level=[80],
            )
            mer = df_te.merge(fc[["unique_id", "ds", "TimeGPT"]], on=["unique_id", "ds"], how="inner")
            if mer.empty:
                reasons[f"empty_merge_h{h}"] += 1
                row[f"smape_h{h}"] = np.nan
                row[f"rmse_h{h}"] = np.nan
            else:
                row[f"smape_h{h}"] = smape(mer["y"], mer["TimeGPT"])
                row[f"rmse_h{h}"] = rmse(mer["y"], mer["TimeGPT"])
        except Exception as e:
            reasons[f"api_error_h{h}"] += 1
            row[f"smape_h{h}"] = np.nan
            row[f"rmse_h{h}"] = np.nan
            skipped_rows.append({"keyword": kw, "reason": f"api_error_h{h}", "info": str(e)})

    metrics_rows.append(row)

# ---------------- Save outputs ----------------
metrics_df = pd.DataFrame(metrics_rows).sort_values("keyword").reset_index(drop=True)
metrics_df.to_csv(OUT_METRICS_CSV, index=False)

skipped_df = pd.DataFrame(skipped_rows)
if not skipped_df.empty:
    skipped_df.to_csv(OUT_SKIPPED_CSV, index=False)

print("\n✅ Done (TimeGPT univariate h=1,6,12).")
print(f"- Metrics CSV: {OUT_METRICS_CSV}")
if not skipped_df.empty:
    print(f"- Skipped log: {OUT_SKIPPED_CSV}")

print("\nReason counts:")
for k, v in reasons.most_common():
    print(f"  {k}: {v}")


In [ ]:
import pandas as pd
import numpy as np

# --- load metrics ---
metrics_path = timegpt_results_dir("semantic_univariate_models") / "timegpt_univariate_metrics_h1_h6_h12.csv"
df = pd.read_csv(metrics_path)

HORIZONS = [1, 6, 12]

rows = []
for h in HORIZONS:
    sm_col = f"smape_h{h}"
    rm_col = f"rmse_h{h}"

    sm = pd.to_numeric(df[sm_col], errors="coerce")
    rm = pd.to_numeric(df[rm_col], errors="coerce")

    rows.append({
        "horizon": h,
        "n_total_keywords": len(df),
        "n_scored": int(sm.notna().sum()),
        "smape_mean": float(sm.mean(skipna=True)),
        "smape_median": float(sm.median(skipna=True)),
        "rmse_mean": float(rm.mean(skipna=True)),
        "rmse_median": float(rm.median(skipna=True)),
    })

summary = pd.DataFrame(rows).sort_values("horizon").reset_index(drop=True)

display(summary)

# Optional: quick look at distribution / NaNs per horizon
nan_report = {
    f"h{h}_smape_nan": int(df[f"smape_h{h}"].isna().sum())
    for h in HORIZONS
}
nan_report.update({
    f"h{h}_rmse_nan": int(df[f"rmse_h{h}"].isna().sum())
    for h in HORIZONS
})
print("NaN counts:", nan_report)
